<a href="https://colab.research.google.com/github/mitulg31/VeriFire/blob/VeriFire_GCollab/VeriFire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip uninstall -y tensorflow-probability &> /dev/null
!pip install tensorflow==2.15.0 keras==2.15.0 tensorflow-probability==0.23.0 onnx onnx-tf==1.10.0 skl2onnx onnxruntime kagglehub &> /dev/null

print("✅ Libraries installed. PLEASE RESTART THE RUNTIME NOW.")
print("Go to the menu: Runtime > Restart runtime (or press Ctrl+M .)")

✅ Libraries installed. PLEASE RESTART THE RUNTIME NOW.
Go to the menu: Runtime > Restart runtime (or press Ctrl+M .)


In [7]:
import kagglehub
import pandas as pd
import numpy as np
import json
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from onnx_tf.backend import prepare

print(f"TensorFlow Version: {tf.__version__}")


# Download the dataset bundle from Kaggle. This returns the path to the downloaded folder.
print("\nDownloading dataset from Kaggle...")
dataset_path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
print(f"Dataset downloaded to: {dataset_path}")

# Construct the full path to the spam.csv file
file_path = os.path.join(dataset_path, "spam.csv")

# Load the CSV file using pandas.
df = pd.read_csv(file_path, encoding='latin-1')

# Prepare the DataFrame
df.rename(columns={'v1': 'label', 'v2': 'message'}, inplace=True)
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
# Drop columns that are sometimes included by mistake in this dataset
df = df[['label', 'message']]

print("\nDataset preview:")
print(df.head())

TensorFlow Version: 2.15.0

Dataset downloaded to: /kaggle/input/sms-spam-collection-dataset

Dataset preview:
   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


In [6]:
# Split the data into features (X) and target (y)
X = df['message']
y = df['label']

# Split data into training and testing sets. `stratify=y` ensures that
# both training and test sets have a similar proportion of spam/ham messages.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create and fit the TF-IDF Vectorizer.
# We limit it to the 5000 most frequent words to keep the model size reasonable.
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Create and train the Multinomial Naive Bayes model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# Evaluate the model's performance on the test set
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Model Accuracy: 97.04%

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       0.99      0.79      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



In [8]:
# --- Part 1: Extract parameters from the Scikit-learn model ---

feature_log_prob = model.feature_log_prob_.T  # Shape: (num_features, num_classes)
class_log_prior = model.class_log_prior_      # Shape: (num_classes,)

# --- Part 2: Build the equivalent model in Keras ---

keras_model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(5000,)),
    tf.keras.layers.Dense(2, name="naive_bayes_layer"),
    tf.keras.layers.Softmax()
])

nb_layer = keras_model.get_layer("naive_bayes_layer")
nb_layer.set_weights([feature_log_prob, class_log_prior])

# Verify the Keras model gives the same predictions as the scikit-learn model
keras_pred_probs = keras_model.predict(X_test_tfidf)
keras_preds = np.argmax(keras_pred_probs, axis=1)
keras_accuracy = accuracy_score(y_test, keras_preds)

print(f"Scikit-learn model accuracy: {accuracy * 100:.2f}%")
print(f"Rebuilt Keras model accuracy: {keras_accuracy * 100:.2f}%")
print("Accuracies match. The Keras model is equivalent.")

# --- Part 3: Convert the Keras model to TensorFlow Lite ---

converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_model_path = "VeriFire_Spam_Model.tflite"
with open(tflite_model_path, "wb") as f:
    f.write(tflite_model)

print(f"\nModel successfully converted to TFLite format at: {tflite_model_path}")


35/35 [==============================] - 2s 5ms/step
Scikit-learn model accuracy: 97.04%
Rebuilt Keras model accuracy: 97.04%
Accuracies match. The Keras model is equivalent.

Model successfully converted to TFLite format at: VeriFire_Spam_Model.tflite


In [10]:
# Get the vocabulary dictionary from the trained vectorizer
vocabulary = tfidf_vectorizer.vocabulary_

serializable_vocabulary = {k: int(v) for k, v in vocabulary.items()}

vocab_path = "tfidf_vocabulary.json"
with open(vocab_path, 'w') as f:
    json.dump(serializable_vocabulary, f)

print(f"Vocabulary with {len(serializable_vocabulary)} words saved to: {vocab_path}")
print("\n--- All steps complete! ---")

Vocabulary with 5000 words saved to: tfidf_vocabulary.json

--- All steps complete! ---
